# Пример 02. Таблица умножения по модулю и точки эллиптической кривой

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** группы вычетов по модулю $n$, обратные элементы в $\mathbb{Z}_n$, эллиптические кривые над конечными полями; связь с криптографией в ML.

## Условие

Требуется построить таблицу умножения по модулю $n=5$ для элементов $\{1,2,3,4\}$, найти обратные элементы в этой группе, а затем — вычислить точку $2G$ на эллиптической кривой над конечным полем.

## Математическая идея

Группа вычетов $\mathbb{Z}_n^{*}$ состоит из элементов $\{1,2,\dots,n-1\}$, взаимно простых с $n$, с операцией умножения по модулю $n$. Элемент $b$ называется обратным к $a$, если

$$a\cdot b \equiv 1 \pmod{n}.$$

Эллиптическая кривая над простым полем $\mathbb{F}_p$ задаётся уравнением

$$y^2 \equiv x^3 + a x + b \pmod{p}.$$

Сложение точек $P+Q$ определяется геометрически: через точку $P$ и $Q$ проводится прямая, третье пересечение с кривой отражается относительно оси $Ox$. Для удвоения точки $P=Q$ используется касательная. Координаты суммы вычисляются по формулам

$$\lambda = \frac{y_2-y_1}{x_2-x_1}\pmod{p},\quad \lambda_{2P} = \frac{3x_1^2+a}{2y_1}\pmod{p},$$

$$x_3 = \lambda^2 - x_1 - x_2,\quad y_3 = \lambda(x_1 - x_3) - y_1.$$

## Решение

1. Для $n=5$ строится таблица умножения $5\times 5$ по модулю.
2. По таблице (или прямым перебором) находятся обратные элементы к $1,2,3,4$.
3. Для кривой $y^2 = x^3 + 2x + 2$ над $\mathbb{F}_{17}$ и точки $G=(5,1)$ вычисляется $2G$ через формулы удвоения.

## Реализация на Python

Функция `mod_mult_table(n)` строит таблицу умножения в $\mathbb{Z}_n$ перебором всех пар элементов. Цикл `for a in [1,2,3,4]` находит обратный элемент через генераторное выражение. Функция `point_add(P, Q)` реализует сложение точек на эллиптической кривой над $\mathbb{F}_p$, причём `pow(2*y1, -1, p)` вычисляет обратный элемент по модулю через встроенную функцию Python 3.8+.


In [1]:
import numpy as np

def mod_mult_table(n):
    elements = np.arange(1, n)
    table = np.zeros((n-1, n-1), dtype=int)
    for i, a in enumerate(elements):
        for j, b in enumerate(elements):
            table[i, j] = (a * b) % n
    return table

print("Таблица умножения для n=5:")
print(mod_mult_table(5))

Таблица умножения для n=5:
[[1 2 3 4]
 [2 4 1 3]
 [3 1 4 2]
 [4 3 2 1]]


In [2]:
for a in [1, 2, 3, 4]:
    inv = next(b for b in [1, 2, 3, 4] if (a * b) % 5 == 1)
    print(f"Обратный для {a}: {inv}")

Обратный для 1: 1
Обратный для 2: 3
Обратный для 3: 2
Обратный для 4: 4


## Дополнительный пример

**Идея.** Эллиптические кривые над конечными полями лежат в основе алгоритмов ECC (Elliptic Curve Cryptography), которые обеспечивают высокий уровень безопасности при малой длине ключа. В контексте ML такие методы применяются для защиты моделей от утечек, конфиденциального вывода и безопасного агрегирования градиентов в федеративном обучении.

**Что демонстрирует код.** Реализуется операция удвоения точки $G\mapsto 2G$ на кривой $y^2 = x^3 + 2x + 2$ над $\mathbb{F}_{17}$. Эта же операция является базовым шагом в алгоритме ECDSA и в протоколах обмена ключами.


In [3]:
p = 17
a, b = 2, 2

def point_add(P, Q):
    if P is None: return Q
    if Q is None: return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and (y1 + y2) % p == 0:
        return None
    if P == Q:
        lam = (3 * x1 * x1 + a) * pow(2 * y1, -1, p) % p
    else:
        lam = (y2 - y1) * pow(x2 - x1, -1, p) % p
    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return(x3, y3)

G = (5, 1)
two_G = point_add(G, G)
print(f"2G = {two_G}")

2G = (6, 3)


## Проверка результата

Корректность таблицы умножения проверяется визуально: каждая ячейка должна содержать остаток от деления произведения на $n$. Корректность обратных элементов проверяется условием $(a\cdot b)\bmod n = 1$. Корректность удвоения точки $2G$ проверяется подстановкой результата в уравнение кривой: координаты $2G$ должны удовлетворять $y^2 \equiv x^3 + 2x + 2 \pmod{17}$.

## Вывод

Конечные группы и эллиптические кривые — это удобный математический аппарат для построения криптографических примитивов, которые применяются, в том числе, для безопасного обучения и развёртывания моделей машинного обучения. Понимание структуры группы точек кривой необходимо для работы с современными протоколами защиты моделей и данных.
